In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

dtype = torch.float16

# ==========================================
# 1. Hyperparameters & Precomputed Constants
# ==========================================
batch_size = 2
dim = 256
n_heads = 4
head_dim = dim // n_heads
vocab_size = 4069
seq_len = 120
max_seq_len = 128

pivot = head_dim // 2
scale = 1.0 / (head_dim ** 0.5)

# ==========================================
# 2. Parameters & Precomputed Buffers
# ==========================================
# Trainable Parameters
w_emb = nn.Parameter(torch.empty((vocab_size, dim), dtype=dtype))
wq = nn.Parameter(torch.empty((dim, dim), dtype=dtype))
wk = nn.Parameter(torch.empty((dim, dim), dtype=dtype))
wv = nn.Parameter(torch.empty((dim, dim), dtype=dtype))
wo = nn.Parameter(torch.empty((dim, dim), dtype=dtype))
w_unemb = nn.Parameter(torch.empty((dim, vocab_size), dtype=dtype))

# RMSNorm Parameters
norm_attn = nn.Parameter(torch.ones(dim, dtype=dtype))
norm_final = nn.Parameter(torch.ones(dim, dtype=dtype))

# Initialize weights (LLaMA standard std=0.02)
for param in [w_emb, wq, wk, wv, wo, w_unemb]:
    nn.init.normal_(param, mean=0.0, std=0.02)

# Precomputed RoPE Frequency Tables (Cos & Sin)
theta = 1.0 / (10000.0 ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
seq_idx = torch.arange(max_seq_len, dtype=torch.float32)
idx_theta = torch.outer(seq_idx, theta)  # Shape: (max_seq_len, pivot)
rope_cos, rope_sin = idx_theta.cos().to(dtype), idx_theta.sin().to(dtype)

# Precomputed Causal Mask Buffer
causal_mask = torch.triu(torch.full((max_seq_len, max_seq_len), float('-inf'), dtype=dtype), diagonal=1)

def rms_norm(x, weight, eps=1e-5):
    # Upcast to float32 for variance to prevent float16 overflow
    variance = x.to(torch.float32).pow(2).mean(dim=-1, keepdim=True)
    return (x * torch.rsqrt(variance + eps).to(dtype)) * weight

def apply_rope(x, cos, sin, pivot=pivot):
    # Autograd-safe 2D rotation via slice assignment (zero torch.cat allocation)
    x1, x2 = x[..., :pivot], x[..., pivot:]
    out = torch.empty_like(x)
    out[..., :pivot] = x1 * cos - x2 * sin
    out[..., pivot:] = x1 * sin + x2 * cos
    return out

# ==========================================
# 3. TRAINING FORWARD & BACKWARD PASS
# ==========================================
tokens = torch.randint(0, vocab_size, (batch_size, seq_len), dtype=torch.long)
targets = torch.randint(0, vocab_size, (batch_size, seq_len), dtype=torch.long)

# 3.1 Input Lookup
x = w_emb[tokens]

# 3.2 Pre-Norm & Q, K, V Projections
x_norm = rms_norm(x, norm_attn)
q = torch.matmul(x_norm, wq).reshape(batch_size, seq_len, n_heads, head_dim).transpose(1, 2)
k = torch.matmul(x_norm, wk).reshape(batch_size, seq_len, n_heads, head_dim).transpose(1, 2)
v = torch.matmul(x_norm, wv).reshape(batch_size, seq_len, n_heads, head_dim).transpose(1, 2)

# 3.3 Apply RoPE to Q and K
q = apply_rope(q, rope_cos[:seq_len], rope_sin[:seq_len])
k = apply_rope(k, rope_cos[:seq_len], rope_sin[:seq_len])

# 3.4 Parallel Causal Attention (Using pre-allocated mask view & scale)
attn = torch.matmul(q, k.transpose(-2, -1)) * scale
attn = torch.softmax(attn + causal_mask[:seq_len, :seq_len], dim=-1)

# 3.5 Output Projection & Residual Connection
out = torch.matmul(attn, v).transpose(1, 2).reshape(batch_size, seq_len, dim)
x = x + torch.matmul(out, wo)

# 3.6 Final Norm & Unembedding
x_norm = rms_norm(x, norm_final)
logits = torch.matmul(x_norm, w_unemb)

# 3.7 Causal Loss & Backpropagation
shift_logits = logits[:, :-1, :].reshape(-1, vocab_size)
shift_targets = targets[:, 1:].reshape(-1)
loss = F.cross_entropy(shift_logits.to(torch.float32), shift_targets)
loss.backward()

print("--- Training Step Completed ---")
print(f"Cross Entropy Loss: {loss.item():.4f}")
print("Gradient computed for wq:", wq.grad.shape)

# ==========================================
# 4. INFERENCE GENERATION STAGE (Inference Mode)
# ==========================================
with torch.inference_mode():
    cache_k = torch.empty((batch_size, n_heads, max_seq_len, head_dim), dtype=dtype)
    cache_v = torch.empty((batch_size, n_heads, max_seq_len, head_dim), dtype=dtype)

    # Write initial prefill keys/values to cache
    cache_k[:, :, :seq_len, :] = k
    cache_v[:, :, :seq_len, :] = v
    next_token = torch.argmax(logits[:, -1, :], dim=-1)

    pos = seq_len
    n_steps = max_seq_len - seq_len

    print("\n--- Generation Loop Started (Inference Mode) ---")
    for step in range(n_steps):
        x = w_emb[next_token].reshape(batch_size, 1, dim)
        x_norm = rms_norm(x, norm_attn)

        q = torch.matmul(x_norm, wq).reshape(batch_size, 1, n_heads, head_dim).transpose(1, 2)
        k = torch.matmul(x_norm, wk).reshape(batch_size, 1, n_heads, head_dim).transpose(1, 2)
        v = torch.matmul(x_norm, wv).reshape(batch_size, 1, n_heads, head_dim).transpose(1, 2)

        q = apply_rope(q, rope_cos[pos:pos + 1], rope_sin[pos:pos + 1])
        k = apply_rope(k, rope_cos[pos:pos + 1], rope_sin[pos:pos + 1])

        cache_k[:, :, pos:pos + 1, :] = k
        cache_v[:, :, pos:pos + 1, :] = v
        pos += 1

        k_past = cache_k[:, :, :pos, :]
        v_past = cache_v[:, :, :pos, :]

        attn = torch.matmul(q, k_past.transpose(-2, -1)) * scale
        attn = torch.softmax(attn, dim=-1)

        out = torch.matmul(attn, v_past).transpose(1, 2).reshape(batch_size, 1, dim)
        x = x + torch.matmul(out, wo)

        x_norm = rms_norm(x, norm_final)
        logits = torch.matmul(x_norm, w_unemb)
        next_token = torch.argmax(logits, dim=-1)

        print(f"Step {step + 1:02d} | Next Token: {next_token.flatten().tolist()}")